In [ ]:
!pip install -q transformers datasets accelerate scikit-learn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

True Tesla T4


In [ ]:
import os
print(os.listdir("/content"))
from google.colab import files
uploaded = files.upload()


['.config', 'drive', 'sample_data']


Saving test.csv to test.csv
Saving train.csv to train.csv
Saving val.csv to val.csv


In [ ]:
import pandas as pd
train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
test = pd.read_csv("test.csv")
print(train.shape, val.shape, test.shape)
print(train.columns.tolist())
print(train.head())
print(train.iloc[:, -1].value_counts())

(75329, 6) (16142, 6) (16142, 6)
['id', 'cleaned_text', 'label_id', 'label_name', 'original_dataset', 'original_label']
      id                                       cleaned_text  label_id  \
0  11878  Its ya boi popsley da 83ar aka the real meek m...         1   
1  35166  [USER] just because a man is gay doesn't mean ...         3   
2  91616  1) 1st u said my edit summary was blank\wrong ...         4   
3  92377  Please don't create personal lists (that is, l...         0   
4  51413  RT [USER] : [USER] You're totally wrong. And I...         4   

          label_name             original_dataset          original_label  
0             Insult         davidson_hate_speech                       1  
1        Hate Speech  cyberbullying_tweets_kaggle                  gender  
2         Harassment        jigsaw_toxic_comments  toxic/severe/obscene=1  
3  Not Cyberbullying        jigsaw_toxic_comments          all_zero_clean  
4         Harassment  cyberbullying_tweets_kaggle     other_c

In [ ]:
print(train.groupby(['label_id', 'label_name']).size())
print(train['cleaned_text'].isna().sum(), "missing texts in train")
print(train['cleaned_text'].str.split().str.len().describe())

label_id  label_name       
0         Not Cyberbullying    20821
1         Insult               18834
2         Threat                 260
3         Hate Speech          18273
4         Harassment           17141
dtype: int64
0 missing texts in train
count    75329.000000
mean        31.288508
std         56.975442
min          1.000000
25%         12.000000
50%         20.000000
75%         32.000000
max       1403.000000
Name: cleaned_text, dtype: float64


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)

def prep(df):
    ds = Dataset.from_pandas(df[["cleaned_text", "label_id"]].rename(columns={"label_id": "labels"}))
    ds = ds.map(lambda b: tok(b["cleaned_text"], truncation=True, max_length=128), batched=True)
    return ds.remove_columns(["cleaned_text"])

train_ds, val_ds, test_ds = prep(train), prep(val), prep(test)
collator = DataCollatorWithPadding(tok)
print(train_ds)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/75329 [00:00<?, ? examples/s]

Map:   0%|          | 0/16142 [00:00<?, ? examples/s]

Map:   0%|          | 0/16142 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 75329
})


In [ ]:
import numpy as np, torch
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

id2label = {0:"Not Cyberbullying", 1:"Insult", 2:"Threat", 3:"Hate Speech", 4:"Harassment"}
label2id = {v:k for k,v in id2label.items()}

# sqrt-inverse-frequency weights (softer than full inverse, avoids unstable training)
counts = train["label_id"].value_counts().sort_index().values
w = 1.0 / np.sqrt(counts)
w = w / w.sum() * len(counts)
class_weights = torch.tensor(w, dtype=torch.float)
print("class weights:", w.round(2))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=5, id2label=id2label, label2id=label2id)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/cyberbullying/out",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=420,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=200,
    report_to="none",
)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds,
                          eval_dataset=val_ds, data_collator=collator,
                          compute_metrics=metrics)
trainer.train()

class weights: [0.38 0.4  3.4  0.41 0.42]


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.500125,0.479468,0.807087,0.737372
2,0.391095,0.477962,0.816442,0.766980
3,0.298923,0.501515,0.820221,0.767270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=7065, training_loss=0.45684225753276636, metrics={'train_runtime': 1387.766, 'train_samples_per_second': 162.842, 'train_steps_per_second': 5.091, 'total_flos': 1.3764007581909978e+16, 'train_loss': 0.45684225753276636, 'epoch': 3.0})

In [ ]:
from sklearn.metrics import classification_report
out = trainer.predict(test_ds)
preds = out.predictions.argmax(-1)
print(classification_report(out.label_ids, preds, target_names=list(id2label.values()), digits=3))

trainer.save_model("/content/drive/MyDrive/cyberbullying/saved_model")
tok.save_pretrained("/content/drive/MyDrive/cyberbullying/saved_model")

                   precision    recall  f1-score   support

Not Cyberbullying      0.829     0.790     0.809      4462
           Insult      0.833     0.868     0.850      4036
           Threat      0.442     0.691     0.539        55
      Hate Speech      0.920     0.891     0.906      3916
       Harassment      0.718     0.744     0.731      3673

         accuracy                          0.823     16142
        macro avg      0.748     0.797     0.767     16142
     weighted avg      0.826     0.823     0.824     16142



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/cyberbullying/saved_model/tokenizer_config.json',
 '/content/drive/MyDrive/cyberbullying/saved_model/tokenizer.json')

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
names = list(id2label.values())
cm = confusion_matrix(out.label_ids, preds)
print(pd.DataFrame(cm, index=names, columns=names))

                   Not Cyberbullying  Insult  Threat  Hate Speech  Harassment
Not Cyberbullying               3527     112       1          119         703
Insult                           128    3503      19          104         282
Threat                             1       9      38            3           4
Hate Speech                      122     214       7         3491          82
Harassment                       476     368      21           77        2731


In [ ]:
test["pred"] = preds
test["wrong"] = test["pred"] != test["label_id"]
print(test.groupby("original_dataset")["wrong"].mean().round(3))
print(test[test.wrong].groupby(["label_name", "original_dataset"]).size().sort_values(ascending=False).head(10))

original_dataset
cyberbullying_tweets_kaggle    0.137
davidson_hate_speech           0.095
jigsaw_toxic_comments          0.213
trac1_aggression               0.415
Name: wrong, dtype: float64
label_name         original_dataset           
Not Cyberbullying  cyberbullying_tweets_kaggle    434
Harassment         jigsaw_toxic_comments          379
                   cyberbullying_tweets_kaggle    293
                   trac1_aggression               270
Not Cyberbullying  trac1_aggression               255
Insult             trac1_aggression               213
Hate Speech        cyberbullying_tweets_kaggle    212
Insult             jigsaw_toxic_comments          198
Not Cyberbullying  jigsaw_toxic_comments          170
Hate Speech        davidson_hate_speech           144
dtype: int64


In [ ]:
mask = test["original_dataset"] != "trac1_aggression"
print("macro F1 without TRAC-1:", f1_score(test.label_id[mask], test.pred[mask], average="macro"))

macro F1 without TRAC-1: 0.7902819504996155
